Installation et téléchargement données

In [ ]:
!pip install wbgapi

import wbgapi as wb
import pandas as pd
from pathlib import Path

countries = ["AUS", "AUT", "BEL", "CAN", "CHL", "COL", "CRI", "CZE", "DNK", "EST", "FIN",
    "FRA", "DEU", "GRC", "HUN", "ISL", "IRL", "ISR", "ITA", "JPN", "KOR",
    "LVA", "LTU", "LUX", "MEX", "NLD", "NZL", "NOR", "POL", "PRT", "SVK",
    "SVN", "ESP", "SWE", "CHE", "TUR", "GBR", "USA"]

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

indicateurs = {
    "wb_chomage": "SL.UEM.TOTL.ZS",
    "wb_pib_hab_ppa": "NY.GDP.PCAP.PP.KD",
    "wb_rnb_hab_ppa": "NY.GNP.PCAP.PP.KD",
}

dfs = {}
for nom, code in indicateurs.items():
    chemin = RAW_DIR / f"{nom}.csv"
    if chemin.exists():
        df = pd.read_csv(chemin, index_col=0)
    else:
        df = wb.data.DataFrame(code, economy=countries, time=range(1991, 2024))
        df.to_csv(chemin)
    dfs[nom] = df.reset_index(names="economy")

taux_chomage = dfs["wb_chomage"]
pib_hab_ppa = dfs["wb_pib_hab_ppa"]
rnb_hab_ppa = dfs["wb_rnb_hab_ppa"]

Mise en forme et rassemblement des données

In [ ]:
import pandas as pd

# On transforme notre tableau afin que les colonnes des années deviennent des valeurs de la colonne "year"

def en_long(df, nom_var):
    """Passe un tableau large (une colonne par année) en format long."""
    long = df.melt(
        id_vars=["economy"],
        var_name="year",
        value_name=nom_var
    )
    return long

df_taux_chomage = en_long(taux_chomage, "Taux chômage")
df_rnb_hab_ppa  = en_long(rnb_hab_ppa, "RNB/hab ppa")
df_pib_hab_ppa  = en_long(pib_hab_ppa, "PIB/hab ppa")


#Tri de nos tableaux par ordre alphabétique des codes des pays 
df_taux_chomage=df_taux_chomage.sort_values(by="economy", ascending = True)
df_rnb_hab_ppa=df_rnb_hab_ppa.sort_values(by="economy", ascending = True)
df_pib_hab_ppa=df_pib_hab_ppa.sort_values(by="economy", ascending = True)



#Création d'un tableau qui associe les codes pays à leurs noms complets en anglais
noms = pd.DataFrame({
    "economy": ["AUS", "AUT", "BEL", "CAN", "CHL", "COL", "CRI", "CZE", "DNK", "EST", "FIN",
    "FRA", "DEU", "GRC", "HUN", "ISL", "IRL", "ISR", "ITA", "JPN", "KOR",
    "LVA", "LTU", "LUX", "MEX", "NLD", "NZL", "NOR", "POL", "PRT", "SVK",
    "SVN", "ESP", "SWE", "CHE", "TUR", "GBR", "USA"],

    "pays": ["Australia", "Austria", "Belgium", "Canada", "Chile", "Colombia", "Costa Rica",
    "Czechia", "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Iceland", "Ireland", "Israel", "Italy", "Japan", "South Korea",
    "Latvia", "Lithuania", "Luxembourg", "Mexico", "Netherlands", "New Zealand",
    "Norway", "Poland", "Portugal", "Slovakia", "Slovenia", "Spain", "Sweden",
    "Switzerland", "Turkey", "United Kingdom", "United States"]
})
#On joint notre tableau pour ajouter les noms des pays, en regroupant par le code pays

df_eco = (df_taux_chomage
          .merge(df_pib_hab_ppa, on=["economy", "year"])
          .merge(df_rnb_hab_ppa, on=["economy", "year"])
          .merge(noms, on="economy", how="left")
          .sort_values(["economy", "year"])
          .reset_index(drop=True))

df_eco = df_eco[["year", "pays", "economy", "Taux chômage", "PIB/hab ppa", "RNB/hab ppa"]]
df_eco.head(10)

Ajout de variables qui vont nous indiquer la variation des variables

In [ ]:
df_eco["Croissance pib/hab"] = df_eco.groupby("economy")["PIB/hab ppa"].pct_change() * 100
df_eco["Variation taux chômage"] = df_eco.groupby("economy")["Taux chômage"].diff()

df_eco.head(10)

Exportation tableau pour le rendre utilisable

In [ ]:
from pathlib import Path

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_eco.to_csv(PROCESSED_DIR / "wb_eco_clean.csv", index=False)

Vérification

In [ ]:
df_eco_verif = pd.read_csv("data/processed/wb_eco_clean.csv")
df_eco_verif.head(10)

Statistiques descriptives 

In [ ]:
#Premier aperçu
df_eco.describe(include="all")

Grâce au tableau des statistiques descriptives ci-dessus, on confirme que nos 38 pays sont bien représentés, nos 33 années de 1991 à 2023 le sont aussi. Il y a bien 38x33=1254 données pour le taux de chômage et le PIB/tête PPA : nos données sont complètes pour ces variables. Néanmoins nous n'avons que 1117 données pour le RNB/tête en PPA, il nous manque des données

Recherche des données manquantes dans la colonne du RNB/hab PPA :

In [ ]:
trous = df_eco[df_eco["RNB/hab ppa"].isna()].groupby("pays")["year"].agg(["min", "max", "count"])
print(trous)

-> beaucoup de pays avec des valeurs manquantes pour le RNB/hab en PPA entre 1991 et 1994, et pour la Turquie toutes manquantes

Répartition des données économiques d'intérêt

On représente la distribution des moyennes des variables, calculées par économie.

In [ ]:
import matplotlib.pyplot as plt

variables = {
    "Taux chômage": "Taux de chômage (%)",
    "PIB/hab ppa": "PIB/hab (PPA)",
    "RNB/hab ppa": "RNB/hab (PPA)",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (col, titre) in zip(axes, variables.items()):
    moyennes = df_eco.groupby("economy")[col].mean()
    ax.hist(moyennes, bins=15, edgecolor="white")
    ax.set_title(f"{titre}\nmoyenne par pays")
    ax.set_xlabel(titre)
    ax.set_ylabel("Nombre de pays")

plt.tight_layout()
plt.show()

On représente l'évolution de la moyenne annuelle de chaque variable.

In [ ]:
import matplotlib.pyplot as plt

variables = {
    "Taux chômage": "Taux de chômage (%)",
    "PIB/hab ppa": "PIB/hab (PPA)",
    "RNB/hab ppa": "RNB/hab (PPA)",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (col, titre) in zip(axes, variables.items()):
    df_eco.groupby("year")[col].mean().plot(ax=ax)
    ax.set_title(f"{titre}\nmoyenne annuelle sur les pays")
    ax.set_xlabel("Année")
    ax.set_ylabel(titre)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Trajectoires selon les pays

In [ ]:
import matplotlib.pyplot as plt

variables = {
    "Taux chômage": "Taux de chômage (%)",
    "PIB/hab ppa": "PIB/hab (PPA)",
    "RNB/hab ppa": "RNB/hab (PPA)",
}
mis_en_avant = ["FRA", "DEU", "USA", "GRC", "KOR", "LVA"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (col, titre) in zip(axes, variables.items()):
    pivot = df_eco.pivot(index="year", columns="economy", values=col)

    # fond : tous les autres pays en gris
    pivot.drop(columns=mis_en_avant).plot(ax=ax, color="lightgrey", legend=False, linewidth=1)
    # premier plan : pays mis en avant, en couleur
    pivot[mis_en_avant].plot(ax=ax, linewidth=2, legend=False)

    ax.set_title(f"{titre} : quelques pays mis en avant")
    ax.set_xlabel("Année")
    ax.set_ylabel(titre)

# une seule légende pour toute la figure
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles[-len(mis_en_avant):], mis_en_avant, title="Pays",
           loc="lower center", ncol=len(mis_en_avant), bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.show()

Corrélation entre les variables

In [ ]:
df_eco[["Taux chômage", "PIB/hab ppa", "RNB/hab ppa"]].corr()